# 09 - Document File Inference

Use trained model weights on a legal document file path.

This notebook is for inference only. It does not add the document to the training dataset and does not retrain the models.

Supported input types: `.txt`, `.pdf`, `.docx`.

## 1. Configure Paths

In [ ]:
from pathlib import Path
import re
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Change this path to your legal document.
DOCUMENT_PATH = PROJECT_ROOT / 'data' / 'inference' / 'your_document.pdf'

SIMPLIFIER_DIR = PROJECT_ROOT / 'models' / 'simplifier'
CLASSIFIER_DIR = PROJECT_ROOT / 'models' / 'clause_classifier'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'document_inference'

MAX_INPUT_LENGTH = 256
MAX_NEW_TOKENS = 128
SIMPLIFIER_BATCH_SIZE = 8
CLASSIFIER_BATCH_SIZE = 16
MIN_CLAUSE_WORDS = 8
MAX_CLAUSE_CHARS = 1600

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Document path: {DOCUMENT_PATH}')
print(f'Simplifier model path: {SIMPLIFIER_DIR}')
print(f'Classifier model path: {CLASSIFIER_DIR}')

## 2. Load Text From File

In [ ]:
def extract_text_from_txt(path: Path) -> str:
    try:
        return path.read_text(encoding='utf-8')
    except UnicodeDecodeError:
        return path.read_text(encoding='latin-1')


def extract_text_from_pdf(path: Path) -> str:
    try:
        import fitz
    except ImportError as exc:
        raise ImportError('PDF inference requires PyMuPDF. Install it with: pip install PyMuPDF') from exc

    chunks = []
    document = fitz.open(path)
    try:
        for page in document:
            chunks.append(page.get_text('text') or '')
    finally:
        document.close()
    return '\n\n'.join(chunks)


def extract_text_from_docx(path: Path) -> str:
    try:
        from docx import Document
    except ImportError as exc:
        raise ImportError('DOCX inference requires python-docx. Install it with: pip install python-docx') from exc

    document = Document(str(path))
    chunks = [paragraph.text.strip() for paragraph in document.paragraphs if paragraph.text.strip()]
    for table in document.tables:
        for row in table.rows:
            cells = [cell.text.strip() for cell in row.cells if cell.text.strip()]
            if cells:
                chunks.append(' | '.join(cells))
    return '\n'.join(chunks)


def extract_text(path: Path) -> str:
    if not path.exists():
        raise FileNotFoundError(f'Document does not exist: {path}')
    if not path.is_file():
        raise ValueError(f'Document path is not a file: {path}')

    suffix = path.suffix.lower()
    if suffix == '.txt':
        return extract_text_from_txt(path)
    if suffix == '.pdf':
        return extract_text_from_pdf(path)
    if suffix == '.docx':
        return extract_text_from_docx(path)
    raise ValueError(f'Unsupported file type: {suffix}. Use .txt, .pdf, or .docx.')


raw_text = extract_text(DOCUMENT_PATH)
print(f'Extracted characters: {len(raw_text)}')
print(raw_text[:1000])

## 3. Clean and Split Into Clauses

In [ ]:
WORD_RE = re.compile(r"[A-Za-z0-9]+(?:[-'][A-Za-z0-9]+)?")
CLAUSE_MARKER_RE = re.compile(
    r"(?=(?:^|\n|\s)(?:section|sec\.?|article|clause|paragraph)\s+\d+(?:\.\d+)*\.?\s+)"
    r"|(?=(?:^|\n|\s)\d+(?:\.\d+)*\.\s+[A-Z])"
    r"|(?=(?:^|\n|\s)\([a-zA-Z0-9]+\)\s+)"
)


def normalize_text(text: object) -> str:
    value = str(text or '').replace('\r\n', '\n').replace('\r', '\n')
    value = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', ' ', value)
    value = re.sub(r'(?<=\w)-\n(?=\w)', '', value)
    value = re.sub(r'[ \t\f\v]+', ' ', value)
    value = re.sub(r'(?im)^\s*page\s+\d+\s*(?:of\s+\d+)?\s*$', '', value)
    value = re.sub(r'\n{3,}', '\n\n', value)
    return value.strip()


def word_count(text: object) -> int:
    return len(WORD_RE.findall(str(text or '')))


def split_long_clause(text: str, max_chars: int = MAX_CLAUSE_CHARS) -> list[str]:
    if len(text) <= max_chars:
        return [text]
    sentences = [piece.strip() for piece in re.split(r'(?<=[.;!?])\s+', text) if piece.strip()]
    chunks = []
    current = ''
    for sentence in sentences or [text]:
        if current and len(current) + len(sentence) + 1 > max_chars:
            chunks.append(current.strip())
            current = sentence
        else:
            current = f'{current} {sentence}'.strip()
    if current:
        chunks.append(current.strip())
    return chunks


def split_into_clauses(text: str) -> list[str]:
    cleaned = normalize_text(text)
    if not cleaned:
        return []

    marker_parts = [part.strip() for part in CLAUSE_MARKER_RE.split(cleaned) if part and part.strip()]
    if len(marker_parts) <= 1:
        marker_parts = [part.strip() for part in re.split(r'\n{2,}', cleaned) if part.strip()]
    if len(marker_parts) <= 1:
        marker_parts = [part.strip() for part in re.split(r'(?<=[.;!?])\s+(?=[A-Z])', cleaned) if part.strip()]

    clauses = []
    for part in marker_parts:
        flat = re.sub(r'\s+', ' ', part).strip()
        if word_count(flat) < MIN_CLAUSE_WORDS:
            continue
        clauses.extend(split_long_clause(flat))
    return clauses


clauses = split_into_clauses(raw_text)
if not clauses:
    raise ValueError('No usable clauses were found in the document.')

clauses_df = pd.DataFrame(
    {
        'clause_number': range(1, len(clauses) + 1),
        'clause_text': clauses,
    }
)
print(f'Prepared clauses: {len(clauses_df)}')
display(clauses_df.head(10))

## 4. Load Trained Models

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoModelForSequenceClassification, AutoTokenizer

from src.classifier import load_label_mapping
from src.simplifier import INPUT_PREFIX

if not (SIMPLIFIER_DIR / 'config.json').exists():
    raise FileNotFoundError(f'Missing trained simplifier model: {SIMPLIFIER_DIR}')
if not (CLASSIFIER_DIR / 'config.json').exists():
    raise FileNotFoundError(f'Missing trained classifier model: {CLASSIFIER_DIR}')
if not (CLASSIFIER_DIR / 'label_mapping.json').exists():
    raise FileNotFoundError(f'Missing classifier label mapping: {CLASSIFIER_DIR / "label_mapping.json"}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'

simplifier_tokenizer = AutoTokenizer.from_pretrained(SIMPLIFIER_DIR)
simplifier_model = AutoModelForSeq2SeqLM.from_pretrained(SIMPLIFIER_DIR).to(device)
simplifier_model.eval()

classifier_tokenizer = AutoTokenizer.from_pretrained(CLASSIFIER_DIR)
classifier_model = AutoModelForSequenceClassification.from_pretrained(CLASSIFIER_DIR).to(device)
classifier_model.eval()
_, id2label = load_label_mapping(CLASSIFIER_DIR / 'label_mapping.json')

print(f'Using device: {device}')
print('Models loaded successfully.')

## 5. Run Simplification and Classification

In [ ]:
def simplify_clauses(texts: list[str]) -> list[str]:
    outputs = []
    for start in range(0, len(texts), SIMPLIFIER_BATCH_SIZE):
        batch = texts[start:start + SIMPLIFIER_BATCH_SIZE]
        prompts = [INPUT_PREFIX + text for text in batch]
        inputs = simplifier_tokenizer(
            prompts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_LENGTH,
        )
        inputs = {key: value.to(device) for key, value in inputs.items()}
        with torch.no_grad():
            generated_ids = simplifier_model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                num_beams=4,
            )
        outputs.extend(simplifier_tokenizer.batch_decode(generated_ids, skip_special_tokens=True))
    return outputs


def classify_clauses(texts: list[str]) -> tuple[list[str], list[float]]:
    labels = []
    confidences = []
    for start in range(0, len(texts), CLASSIFIER_BATCH_SIZE):
        batch = texts[start:start + CLASSIFIER_BATCH_SIZE]
        inputs = classifier_tokenizer(
            batch,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_LENGTH,
        )
        inputs = {key: value.to(device) for key, value in inputs.items()}
        with torch.no_grad():
            logits = classifier_model(**inputs).logits
            probabilities = torch.softmax(logits, dim=-1)
            batch_confidences, predicted_ids = probabilities.max(dim=-1)
        labels.extend(id2label[int(index)] for index in predicted_ids.detach().cpu())
        confidences.extend(round(float(value), 6) for value in batch_confidences.detach().cpu())
    return labels, confidences


clause_texts = clauses_df['clause_text'].tolist()
simplified_clauses = simplify_clauses(clause_texts)
predicted_types, classifier_confidences = classify_clauses(clause_texts)

result_df = clauses_df.copy()
result_df['simplified_clause'] = simplified_clauses
result_df['predicted_clause_type'] = predicted_types
result_df['classifier_confidence'] = classifier_confidences

display(result_df.head(20))

## 6. Save Outputs

In [ ]:
output_stem = DOCUMENT_PATH.stem
csv_path = OUTPUT_DIR / f'{output_stem}_model_outputs.csv'
txt_path = OUTPUT_DIR / f'{output_stem}_model_outputs.txt'

result_df.to_csv(csv_path, index=False)

lines = [
    'LegalEase document inference output',
    'Educational assistance only; this system does not provide legal advice.',
    f'Source document: {DOCUMENT_PATH}',
    '',
]
for _, row in result_df.iterrows():
    lines.append(f"Clause {row['clause_number']}")
    lines.append(f"Original: {row['clause_text']}")
    lines.append(f"Simplified: {row['simplified_clause']}")
    lines.append(f"Predicted clause type: {row['predicted_clause_type']} ({row['classifier_confidence']})")
    lines.append('')

txt_path.write_text('\n'.join(lines), encoding='utf-8')

print(f'Saved CSV: {csv_path}')
print(f'Saved TXT: {txt_path}')